# Comparison of GEMSS vs. ALFESE

Analysis of comparative benchmark results. Datasets were generated using native GEMSS benchmarking framework.

In [ ]:
from IPython.display import display, Markdown
import yaml
import pandas as pd
import numpy as np
import plotly.io as pio
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

# display plotly figures in notebook
pio.renderers.default = "notebook"

In [ ]:
config_path = Path("../configs/benchmark_config.yaml")

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

# Extract experiment information
exp_setup = []
for exp in config["experiments"]:
    exp_setup.append(
        {
            "Exp": exp["exp_number"],
            "Description": exp["description"],
            "N": exp["dataset"]["n_samples"],
            "P": exp["dataset"]["n_features"],
            "Noise": exp["dataset"]["noise_std"],
            "Missing": exp["dataset"]["nan_ratio"],
            "Class Ratio": exp["dataset"]["binary_response_ratio"],
        }
    )

exp_setup_df = pd.DataFrame(exp_setup)

display(Markdown("### Experiment Setup Overview"))
display(Markdown(f"**Total experiments**: {len(exp_setup_df)}"))
display(exp_setup_df)


## Loading Benchmark Results

In [ ]:
# "sum" or "latest"
# - "sum" will load the cumulative results file 'benchmark_results_sum.csv'
# - "latest" will load the most recent file in the results directory
file_mode = "latest"

In [ ]:
# Find the results file
results_dir = Path("../results")
csv_files = list(results_dir.glob("benchmark_results_*.csv"))

if not csv_files:
    display(Markdown("❌ **No results files found in ../results/**"))
    display(
        Markdown("Run the benchmark first: `uv run python scripts/run_benchmark.py`")
    )
    df = None
else:
    if file_mode == "latest":
        # Get most recent file
        results_file = max(csv_files, key=lambda p: p.stat().st_mtime)
    elif file_mode == "sum":
        results_file = results_dir / "benchmark_results_sum.csv"

    # Load data
    df = pd.read_csv(results_file)

    display(Markdown(f"✅ **Loaded {len(df)} results from `{results_file.name}`**"))

    # Display experiments
    if "Exp_Number" in df.columns:
        exp_count = df["Exp_Number"].nunique()
        exp_numbers = sorted(df["Exp_Number"].unique().tolist())
        display(
            Markdown(
                f"**{exp_count} Experiments**: {', '.join(map(str, exp_numbers))}"
            )
        )

    methods = ", ".join(sorted(df["Method"].unique().tolist()))
    display(Markdown(f"**Methods**: {methods}"))
    status_counts = dict(df["Status"].value_counts())
    display(Markdown(f"**Status**: {status_counts}"))

In [ ]:
# sort df by Exp_Number and Method
if df is not None:
    df = df.sort_values(by=["Exp_Number", "Method"]).reset_index(drop=True)

# format recall, precision and F1 score to 3 decimals
for col in ["Recall", "Precision", "F1_Score", "Jaccard"]:
    df[col] = df[col].astype(float).round(3)

for col in ["Adjusted_SI", "Success_Index", "Runtime_Sec"]:
    df[col] = df[col].astype(float).round(1)

In [ ]:
identifiers = ["Exp_Number", "Experiment_ID", "Method", "Experiment_Description"]
# Metrics ordered by importance: F1 Score (most important), ASI, then others
metrics = [
    "F1_Score",
    "Adjusted_SI",
    "Recall",
    "Precision",
    "Success_Index",
    "Runtime_Sec",
]

exp_number2description = dict(
    set(zip(df["Exp_Number"], df["Experiment_Description"], strict=True))
)

In [ ]:
# Define consistent color palette for all methods
METHOD_COLORS = {
    "GEMSS": "#000000",  # Black
    "ALFESE_mi_tau1.0": "#2ca02c",  # Green
    "ALFESE_mi_tau0.5": "#9c9c08",  # Olive
    "ALFESE_greedy_tau1.0": "#5f2199",  # Purple
    "ALFESE_greedy_tau0.5": "#b15dcb",  # Purple
    "ALFESE_importance_tau1.0": "#2741d6",  # Blue
    "ALFESE_importance_tau0.5": "#279cd6",  # Blue
    "ALFESE_mrmr_tau1.0": "#dd3708",  # Red
    "ALFESE_mrmr_tau0.5": "#ff6e0e",  # Orange
    "ALFESE_fcbf_tau1.0": "#8c564b",  # Brown
    "ALFESE_fcbf_tau0.5": "#c49c94",  # Light brown
}

In [ ]:
# count failed and successful experiments per method
all_methods = sorted(df["Method"].unique())
success_counts = df[df["Status"] == "Success"].groupby("Method").size()
failed_counts = df[df["Status"] != "Success"].groupby("Method").size()

# Reindex to show all methods
success_counts = success_counts.reindex(all_methods, fill_value=0)
failed_counts = failed_counts.reindex(all_methods, fill_value=0)

# Combine into a dataframe
status_summary = pd.DataFrame(
    {"Successful_Count": success_counts, "Failed_Count": failed_counts}
)

display(Markdown("## Experiment Status Count by Method"))
display(status_summary)

In [ ]:
# Filter successful results globally
df_success = df[df["Status"] == "Success"].copy()

all_experiments = sorted(df["Exp_Number"].unique())

for exp_number in all_experiments:
    df_exp = df[df["Exp_Number"] == exp_number]
    df_exp_success = df_exp[df_exp["Status"] == "Success"]

    # Get experiment ID for display
    exp_id = df_exp["Experiment_ID"].iloc[0] if len(df_exp) > 0 else "Unknown"

    display(Markdown(f"## 🔍 Experiment {exp_number}: `{exp_id}`"))
    display(Markdown(f"{exp_number2description.get(exp_number, '')}"))

    if df_exp_success.empty:
        display(Markdown(f"⚠️ **No successful runs for experiment {exp_number}**"))
    else:
        display(Markdown(f"### Performance Metrics Summary:"))
        stats = df_exp_success[["Method"] + metrics]
        display(stats)

## Performance Metrics Across Experiments

Visualize method comparison across experiments.

In [ ]:
# Create a figure with subplots for each metric
fig = make_subplots(
    rows=3,
    cols=2,
    subplot_titles=(
        "F1 Score",
        "Runtime",
        "Recall",
        "Precision",
        "Adjusted Success Index",
        "Success Index",
    ),
    vertical_spacing=0.12,
    horizontal_spacing=0.1,
)

# Plot each metric - prioritizing F1 Score and ASI
metrics_info = [
    ("F1_Score", 1, 1, [0, 1.1], "linear"),
    ("Runtime_Sec", 1, 2, None, "log"),
    ("Recall", 2, 1, [0, 1.1], "linear"),
    ("Precision", 2, 2, [0, 1.1], "linear"),
    ("Adjusted_SI", 3, 1, None, "linear"),
    ("Success_Index", 3, 2, None, "linear"),
]

for metric, row, col, y_range, y_type in metrics_info:
    # Get all experiment numbers across the entire dataset
    all_exp_numbers = sorted(df["Exp_Number"].unique())

    for method in df["Method"].unique():
        method_data = df[df["Method"] == method].copy()

        # Reindex to include all experiments, filling missing with NaN
        method_data = method_data.set_index("Exp_Number").reindex(all_exp_numbers)

        # Convert to lists - NaN becomes None for plotly, creating gaps
        x_values = method_data.index.tolist()
        y_values = method_data[metric].tolist()

        fig.add_trace(
            go.Scatter(
                x=x_values,
                y=y_values,
                mode="lines+markers",
                name=method,
                legendgroup=method,
                showlegend=(row == 1 and col == 1),  # Only show legend once
                marker=dict(size=8, color=METHOD_COLORS.get(method, "#3b676c")),
                line=dict(width=2, color=METHOD_COLORS.get(method, "#3b676c")),
                connectgaps=False,  # Don't connect across missing data
            ),
            row=row,
            col=col,
        )

    # Update axes
    fig.update_xaxes(title_text="Experiment Number", dtick=1, row=row, col=col)
    fig.update_yaxes(title_text=metric.replace("_", " "), type=y_type, row=row, col=col)
    if y_range:
        fig.update_yaxes(range=y_range, row=row, col=col)

# Update layout with increased top margin for title and legend
fig.update_layout(
    height=1400,
    showlegend=True,
    legend=dict(
        orientation="h",
        yanchor="top",
        y=1.08,  # Position legend above the plots
        xanchor="right",
        x=1,
        font=dict(size=14)
    ),
    title_text="Performance Metrics Across All Experiments",
    title_y=0.97,  # Position title at the top of the expanded margin
    margin=dict(t=180),  # Increase top margin to make room for title and legend
)

fig.show()

## Best Performers by Experiment

Identify the best method for each experiment based on F1 Score.

In [ ]:
display(Markdown("### Best Method per Experiment (by F1 Score)"))

# Find best method for each experiment
best_per_exp = []
for exp_num in sorted(df["Exp_Number"].unique()):
    exp_data = df[df["Exp_Number"] == exp_num]
    best_row = exp_data.loc[exp_data["F1_Score"].idxmax()]

    best_per_exp.append(
        {
            "Exp_Number": int(best_row["Exp_Number"]),
            "Experiment_ID": best_row["Experiment_ID"],
            "Best_Method": best_row["Method"],
            "F1_Score": round(best_row["F1_Score"], 3),
            "Adjusted_SI": round(best_row["Adjusted_SI"], 3),
            "Recall": round(best_row["Recall"], 3),
            "Precision": round(best_row["Precision"], 3),
            "Runtime_Sec": round(best_row["Runtime_Sec"], 1),
        }
    )

best_df = pd.DataFrame(best_per_exp)
display(best_df)

# Overall method winning statistics
display(Markdown("\n### Method Win Statistics (across all experiments, F1 Score)"))

# Get all methods and their win counts (including 0s)
# Reverse sort to show GEMSS first
all_methods = sorted(df["Method"].unique(), reverse=True)
method_wins = best_df["Best_Method"].value_counts().reindex(all_methods, fill_value=0)
win_stats = pd.DataFrame(
    {
        "Method": method_wins.index,
        "Experiments_Won": method_wins.values,
        "Win_Rate": (method_wins.values / len(best_df) * 100).round(1),
    }
)
display(win_stats)

# Visualize method wins and runtime comparison
fig_comparison = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "Experiments Won (F1 Score)",
        "Average Runtime by Method",
    ),
    horizontal_spacing=0.15,
)

# Left plot: Experiments won
fig_comparison.add_trace(
    go.Bar(
        x=win_stats["Method"],
        y=win_stats["Experiments_Won"],
        marker_color=[METHOD_COLORS.get(m, "#3b676c") for m in win_stats["Method"]],
        text=win_stats["Win_Rate"].apply(lambda x: f"{x}%"),
        textposition="outside",
        showlegend=False,
    ),
    row=1,
    col=1,
)

# Right plot: Average runtime by method (keep same order as other plots)
runtime_stats = (
    df.groupby("Method")["Runtime_Sec"].mean().reindex(all_methods).reset_index()
)

fig_comparison.add_trace(
    go.Bar(
        x=runtime_stats["Method"],
        y=runtime_stats["Runtime_Sec"],
        marker_color=[METHOD_COLORS.get(m, "#3b676c") for m in runtime_stats["Method"]],
        text=runtime_stats["Runtime_Sec"].apply(lambda x: f"{x:.1f}s"),
        textposition="outside",
        showlegend=False,
    ),
    row=1,
    col=2,
)

# Update layout
fig_comparison.update_xaxes(title_text="Method", row=1, col=1)
fig_comparison.update_yaxes(title_text="No. Experiments Won", row=1, col=1)
fig_comparison.update_xaxes(title_text="Method", row=1, col=2)
fig_comparison.update_yaxes(title_text="Runtime (seconds)", row=1, col=2)

fig_comparison.update_layout(
    height=500,
    title_text="Method Performance Comparison: F1 Score Wins vs Runtime",
    showlegend=False,
)

fig_comparison.show()

---

## Notes

### Methodology:

This benchmark compares feature selection methods using **synthetically generated datasets with known ground truth**. 

**Why Synthetic Data?**
- **Known Ground Truth**: True feature sets are known by design, enabling precise evaluation
- **Controlled Properties**: Each dataset is generated with specific parameters (sample size, feature count, noise, missing data, class imbalance)
- **Reproducibility**: Seeded generation ensures consistent results across runs
- **Systematic Testing**: Multiple experiments across different tiers isolate specific challenges:
  - **Tier 1**: Basic validation on clean data
  - **Tier 2**: High-dimensional stress test (P ≥ 1000)
  - **Tier 4**: Robustness under adversity (high noise, missing data)
  - **Tier 7**: Class imbalance scenarios

The tiered approach is inherited from the validation of GEMSS and all the experiments are taken from the original test suite (except for the increase of minimal sample size $n=25$ to $n=30$ due to limitations of ALFESE.)

**Methods Compared:**
- **GEMSS**: Multi-solution feature selector using Bayesian framework.
- **ALFESE** (multiple variants): Knockoff-based method with different feature scorers (mrmr, mi, fcbf) and thresholds.

**Evaluation Approach:**
- Each method returns one or more feature sets (candidate solutions)
- The **union** of all selected features is compared against the ground truth
- Metrics quantify recovery (Recall), accuracy (Precision), and overall quality (F1, Adjusted SI)

### Metrics Explained:

- **F1 Score** (Most Important): Harmonic mean of precision and recall, balances both metrics (higher is better)
  $$F1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$
- **Adjusted SI**: Success Index weighted by precision, combines recall and precision (higher is better)
  $$\text{Adjusted SI} = \text{SI} \cdot \text{Precision}$$
- **Recall**: Fraction of true features successfully recovered (higher is better)
  $$\text{Recall} = \frac{\text{correct features}}{\text{total true features}}$$
- **Precision**: Fraction of selected features that are true positives (higher is better)
  $$\text{Precision} = \frac{\text{correct features}}{\text{total selected features}}$$
- **Success Index (SI)**: GEMSS-specific metric balancing coverage and accuracy
  $$\text{SI} = \frac{P \cdot \text{correct}}{(\text{true features})^2}$$
